ETL para disponibilizar os dados para feature engineering

In [0]:
import zipfile
import os
import shutil


# caminho onde esta o arquivo zip        
zip_path = "/Volumes/churn_tables/bronze/dados_brutos/bases.zip.zip"
extract_to_local = "/tmp/extraidos_bases/"

# cria a pasta temporária local
os.makedirs(extract_to_local, exist_ok=True)

# descompacta o arquivo para a pasta temporária local
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_local)

print("arquivos descompactados com sucesso no diretorio temporario")


In [0]:
arquivos = os.listdir("/tmp/extraidos_bases/bases/")
print(arquivos)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS churn_tables.silver;

In [0]:
# criacao de catalogo, schema e volume
spark.sql("CREATE CATALOG IF NOT EXISTS churn_tables;")
spark.sql("CREATE SCHEMA IF NOT EXISTS churn_tables.silver;")
spark.sql("CREATE VOLUME IF NOT EXISTS churn_tables.silver.dados_csv")

origem_path = "/tmp/extraidos_bases/bases/"
volume_path = "/Volumes/churn_tables/silver/dados_csv/"

delimitadores = {
    'clientes.csv': ';',
    'produtos.csv': ';',
    'transacoes.csv': ',',
    'transacao_produto.csv': ','
}

# copia os arquivos do drive local para o catalog
for tab in delimitadores.keys():
    origem_arquivo = os.path.join(origem_path, tab)
    destino_arquivo = os.path.join(volume_path, tab)
    
    if os.path.exists(origem_arquivo):
        shutil.copy(origem_arquivo, destino_arquivo)
        print(f"arquivo {tab} copiado para o volume.")
    else:
        print(f"arquivo: {tab} nao foi encontrado em {origem_path}")

# le do bronze e salva no silver 
for tab, delim in delimitadores.items():
    nome_tabela = tab.replace('.csv', '')

    df = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("delimiter", delim) 
          .load(f"{volume_path}{tab}"))

    print(f"\n--- amostra da tabela: {nome_tabela} ---")
    df.show(5)
    
    df.write.mode("overwrite").saveAsTable(f"churn_tables.silver.{nome_tabela}")
    print(f"tabela churn_tables.silver.{nome_tabela} salva com sucesso!")